In [0]:
# Required Imports
import uuid
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalog", "azdeusdbricks")
dbutils.widgets.text("schema", "bronze")
dbutils.widgets.text("table", "raw_files")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table = dbutils.widgets.get("table")


In [0]:
src_table = f"{catalog}.{schema}.{table}"
dq_table = f"{catalog}.{schema}.dq_results"

run_id = str(uuid.uuid4())
run_ts = datetime.utcnow()


In [0]:
df = spark.table(src_table)
total_rows = df.count()

results = []



In [0]:
def add_results(name, severity, failed, total=None):
    total = total_rows if total is None else total
    if failed == 0:
        status = "PASS"
    else:
        status = "FAIL" if severity == "ERROR" else "WARN"

    results.append({
        "run_id":run_id,
        "run_ts": run_ts,
        "table_name": src_table,
        "check_name": name,
        "severity": severity,
        "failed_count": int(failed),
        "total_count": int(total),
        "failed_pct": round(100.0* failed/total, 2) if total else 0.0,
        "status": status
    })

In [0]:
# table is not empty
add_results("table_not_empty", "ERROR", 1 if total_rows == 0 else 0, total = 1)

expected_columns = [
"record_id", "order_id", "order_date", "customer_name", "customer_email", "phone_number", "city", "country", "product_name", "category", "quantity", "unit_price", "discount_percent", "payment_method", "order_status", "rating", "_rescued_data", "_ingestTimeStamp", "sourceFileLocation"
]

missing_columns = [c for c in expected_columns if c not in df.columns]

add_results("expected_columns_present", "ERROR", len(missing_columns), total = len(expected_columns))

if missing_columns:
    print(f"Missing columns: {missing_columns}")




In [0]:
date_formats = ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MM-yyyy", "yyyy/MM/dd", "MMM d, yyyy"]

parsed_date = "coalesce("+", ".join(
    f"try_to_timestamp(trim(order_date), '{f}')" for f in date_formats
)+ ")"


row_checks = [
    ("record_id_not_null", "ERROR", "record_id IS NULL"),
    ("order_id_not_null", "ERROR", "order_id IS NULL"),
    ("no_rescued_data", "ERROR", "_rescued_data IS NOT NULL"),

    ("order_date_parseble", "WARN", f"order_date is NULL OR {parsed_date} IS NULL"),
    ("order_date_not_future", "WARN", f"{parsed_date} > current_timestamp()"),
    ("customer_name_valid", "WARN", "customer_name IS NULL OR trim(customer_name) = '' OR  upper(trim(customer_name)) = 'NULL'"),
    ('customer_email_valid', "WARN", "customer_email IS NULL OR NOT (trim(customer_email) RLIKE '^[A-Za-z0-9.%+-]+@[A-Za-z]{2,}$')"),
    ("phone_valid", "WARN", "phone number IS NULL OR trim(phone_number)= '' OR regex p_replace(phone_number, '^[0-9], '')RLIKE '^0+$'"),
    ("city_valid", "WARN", "city IS NULL OR trim(city) = '' OR  upper(trim(city)) IN ('N/A', 'NA', 'NULL')"),
    ("quantity_positive", "WARN", "try_cast(quantity AS INT) IS NULL OR try_cast(quantity AS INT) <= 0"),
    ("unit_price_positive", "WARN", "try_cast(unit_price AS DOUBLE) IS NULL OR try_cast(unit_price AS DOUBLE) <=0"),
    ("discount_in_0_100", "WARN",  "try_cast(discount_percent AS DOUBLE) IS NULL OR try)cast(discount_percent AS DOUBLE) < 0 OR try_cast(discount_percent AS DOUBLE) > 100"),
    ("rating_in_1_5", "WARN", "try_cast(rating AS INT) IS NOT NULL AND (try_cast(rating AS INT) < 1 try_cast(rating AS INT) > 5)"),
    ("order_status_valid", "WARN", "order_status IS NULL or lower(trim(order_status)) NOT IN ('delivered', 'shipped', 'processing', 'cancelled', 'returned')"),
    ("category_is_standard", "WARN", "category IS NULL OR category != initcap(trim(category))"),


]